### Get Imports

In [ ]:
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

MODEL = 'gpt-4.1-mini'
openai = OpenAI()


### Check if API key is accessible 

In [2]:
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    

API key looks good so far


### System and User Prompt for getting relevant links from the website

In [3]:
link_system_prompt ="""
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about a company,
such as links to an About page, or a Company page, or Careers/Jobs page.
You should respond in JSON as in this example:

{
    "links" : [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company,
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

print(get_links_user_prompt("https://huggingface.co"))


Here is the list of links on the website https://huggingface.co -
Please decide which of these are relevant web links for a brochure about the company,
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

/
/models
/datasets
/spaces
/storage
/docs
/enterprise
/pricing
/tasks
/chat
/collections
/languages
/organizations
/blog
/posts
/papers
/hardware
/learn
/join/discord
https://discuss.huggingface.co/
https://github.com/huggingface
/enterprise
/pro
/support
/inference/models
/inference-endpoints
/storage
/login
/join
/spaces
/models
/Qwen/Qwen3.8-27B
/unsloth/Qwen3.8-27B-GGUF
/Qwen/Qwen3.8-2.4T-A95B
/Lightricks/LTX-2.5
/MiniMaxAI/MiniMax-Music3
/models
/spaces/MiniMaxAI/MiniMax-Music3
/spaces/MiniMaxAI/MiniMax-H3-Turbo-Lora
/spaces/prithivMLmods/Qwen-Image-Edit-2511-LoRAs-Fast
/spaces/agent-memory-leaderboard/leaderboard
/spaces/thornmaze/reel-lab
/spaces
/datasets/r0b0tlab/qwen3.8-max-glm5.2-kim

### Function to retrive relevant links from the website using the model

In [5]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model = MODEL,
        messages = [
            {"role": "system", "content" : link_system_prompt},
            {"role": "user", "content" : get_links_user_prompt(url)}
        ]
    )
    results = response.choices[0].message.content
    links = json.loads(results)
    return links

select_relevant_links("https://huggingface.co")

{'links': [{'type': 'home page', 'url': 'https://huggingface.co/'},
  {'type': 'brand page', 'url': 'https://huggingface.co/brand'},
  {'type': 'blog', 'url': 'https://huggingface.co/blog'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'},
  {'type': 'endpoints product', 'url': 'https://endpoints.huggingface.co'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'GitHub', 'url': 'https://github.com/huggingface'},
  {'type': 'Twitter', 'url': 'https://twitter.com/huggingface'},
  {'type': 'LinkedIn', 'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'Discussions', 'url': 'https://discuss.huggingface.co/'},
  {'type': 'Join Discord', 'url': 'https://huggingface.co/join/discord'}]}

### Function to get all information regarding the webpage 

In [ ]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

print(fetch_page_and_all_relevant_links("https://huggingface.co"))